In [ ]:
from langgraph.graph import StateGraph,START, END
from typing import TypedDict, List,Literal,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel,Field
from langchain_google_genai import ChatGoogleGenerativeAI
import google.genai as genai
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage,HumanMessage,BaseMessage,AIMessage
from langgraph.checkpoint.memory import InMemorySaver,MemorySaver

load_dotenv()
import os

In [ ]:
class JokeState(TypedDict):
    topic : str
    joke : str
    explanation : str

    

In [ ]:
client = genai.Client(
    vertexai=True, 
    project=os.getenv('GEMINI_API_KEY'), 
    location='us-central1'
)

In [ ]:
def generate_joke(state:JokeState):

    prompt = f"generate joke on given topic {state['topic']}"
    response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt
     ).text
    
    return {'joke' : response}



In [ ]:
def generate_explanation(state:JokeState):

    prompt = f"explain the joke {state['joke']} "
    response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt
     ).text
    
    return {'explanation' : response}

In [ ]:
graph  = StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer  = InMemorySaver()

workflow = graph.compile(checkpointer= checkpointer)

#Displaying the graph
print(workflow.get_graph().draw_ascii())

      +-----------+      
      | __start__ |      
      +-----------+      
            *            
            *            
            *            
    +---------------+    
    | generate_joke |    
    +---------------+    
            *            
            *            
            *            
+----------------------+ 
| generate_explanation | 
+----------------------+ 
            *            
            *            
            *            
      +---------+        
      | __end__ |        
      +---------+        


In [ ]:
config1 = {'configurable':{'thread_id':1}}
workflow.invoke({'topic':'pizza'},config=config1)

c:\Users\H604660\Desktop\Siddhi Shintre\My Learnings\LANGGRAPH\myenv\Lib\site-packages\google\auth\_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


{'topic': 'pizza',
 'joke': 'Why did the pizza maker get fired? \n\nBecause he kept topping people off!\n',
 'explanation': 'The joke plays on the double meaning of the word "topping":\n\n*   **Literal Meaning:** In the context of pizza making, "topping" refers to the ingredients you put on top of the pizza crust (like cheese, pepperoni, vegetables).\n*   **Figurative Meaning:** "To top someone off" means to give someone a drink or a serving of something until their glass or plate is full. It can also imply generosity or being overly generous.\n\nThe humor comes from the unexpected shift in meaning. We initially think the joke will be about a pizza-making mistake related to ingredients, but it turns out to be a pun about the pizza maker being too generous with something (implied to be drinks or servings), likely outside of their job description, to the point where it became inappropriate and got them fired.\n'}

In [ ]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza maker get fired? \n\nBecause he kept topping people off!\n', 'explanation': 'The joke plays on the double meaning of the word "topping":\n\n*   **Literal Meaning:** In the context of pizza making, "topping" refers to the ingredients you put on top of the pizza crust (like cheese, pepperoni, vegetables).\n*   **Figurative Meaning:** "To top someone off" means to give someone a drink or a serving of something until their glass or plate is full. It can also imply generosity or being overly generous.\n\nThe humor comes from the unexpected shift in meaning. We initially think the joke will be about a pizza-making mistake related to ingredients, but it turns out to be a pun about the pizza maker being too generous with something (implied to be drinks or servings), likely outside of their job description, to the point where it became inappropriate and got them fired.\n'}, next=(), config={'configurable': {'thread_id': '1', 'ch

In [ ]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza maker get fired? \n\nBecause he kept topping people off!\n', 'explanation': 'The joke plays on the double meaning of the word "topping":\n\n*   **Literal Meaning:** In the context of pizza making, "topping" refers to the ingredients you put on top of the pizza crust (like cheese, pepperoni, vegetables).\n*   **Figurative Meaning:** "To top someone off" means to give someone a drink or a serving of something until their glass or plate is full. It can also imply generosity or being overly generous.\n\nThe humor comes from the unexpected shift in meaning. We initially think the joke will be about a pizza-making mistake related to ingredients, but it turns out to be a pun about the pizza maker being too generous with something (implied to be drinks or servings), likely outside of their job description, to the point where it became inappropriate and got them fired.\n'}, next=(), config={'configurable': {'thread_id': '1', 'c

In [ ]:
config2 = {'configurable':{'thread_id':2}}
workflow.invoke({'topic':'pasta'},config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti blush?\n\nBecause it saw the meat sauce!\n',
 'explanation': 'The joke relies on a double entendre (a phrase with two meanings, one of which is usually suggestive or humorous).\n\n*   **Literal meaning:** Spaghetti is a type of pasta, and meat sauce is a common topping for it. So, in a simple sense, the joke describes spaghetti encountering its usual accompaniment.\n\n*   **Implied (adult) meaning:** "Meat" can be a slang term for the male anatomy, and "sauce" can imply sexual activity or fluids. The blush implies embarrassment or arousal. Therefore, the joke humorously suggests that the spaghetti is blushing because it is encountering a suggestive situation with the "meat sauce."\n'}

In [ ]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti blush?\n\nBecause it saw the meat sauce!\n', 'explanation': 'The joke relies on a double entendre (a phrase with two meanings, one of which is usually suggestive or humorous).\n\n*   **Literal meaning:** Spaghetti is a type of pasta, and meat sauce is a common topping for it. So, in a simple sense, the joke describes spaghetti encountering its usual accompaniment.\n\n*   **Implied (adult) meaning:** "Meat" can be a slang term for the male anatomy, and "sauce" can imply sexual activity or fluids. The blush implies embarrassment or arousal. Therefore, the joke humorously suggests that the spaghetti is blushing because it is encountering a suggestive situation with the "meat sauce."\n'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1099c6-3e45-645d-8002-7c3095396092'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-02-14T11:57:52.065203+00:00', p

In [ ]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti blush?\n\nBecause it saw the meat sauce!\n', 'explanation': 'The joke relies on a double entendre (a phrase with two meanings, one of which is usually suggestive or humorous).\n\n*   **Literal meaning:** Spaghetti is a type of pasta, and meat sauce is a common topping for it. So, in a simple sense, the joke describes spaghetti encountering its usual accompaniment.\n\n*   **Implied (adult) meaning:** "Meat" can be a slang term for the male anatomy, and "sauce" can imply sexual activity or fluids. The blush implies embarrassment or arousal. Therefore, the joke humorously suggests that the spaghetti is blushing because it is encountering a suggestive situation with the "meat sauce."\n'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1099c6-3e45-645d-8002-7c3095396092'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-02-14T11:57:52.065203+00:00', 

In [ ]:
### TIME TRAVEL

graph.get_state({'configurable': {'thread_id':2,'checkpoint_id': '1f1099c6-18f4-6504-8000-47bedf084151'}})